# 01 · DFL-Spiele laden und Labels ableiten

Training und Tuning sind nur auf externen Spielen erlaubt. Dieses Notebook liest die sieben offenen DFL/IDSSE-Spiele (CC BY 4.0) und speichert je Frame Halbzeit, Label (`H`, `A`, `D`), Ball und alle Spielerpositionen. `D` kommt aus dem DFL-Ballstatus, `H`/`A` aus dem DFL-Teambesitz.

**Ausgabe:** `artifacts/dfl/<Spiel>.parquet`, Eingabe für Notebook 02.

In [1]:
# Standard library
import json
import time
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Tracking loader
import kloppy
from kloppy import sportec

ROOT = Path.cwd().resolve().parent
RAW = ROOT / "external_data" / "dfl_idsse" / "raw"
OUT = ROOT / "artifacts" / "dfl"
OUT.mkdir(parents=True, exist_ok=True)
FPS = 25

print(f"pandas {pd.__version__} · numpy {np.__version__} · kloppy {kloppy.__version__}")

pandas 2.3.3 · numpy 1.26.4 · kloppy 3.18.0


## 1. Laden

kloppy liest Positionen und Matchinfo einschließlich der Frames außerhalb des laufenden Spiels (`only_alive=False`). Spieler werden nach Team präfixiert (`H_…`, `A_…`). Bereits gespeicherte Spiele werden wiederverwendet.

In [2]:
manifest = pd.DataFrame(json.load(open(RAW.parent / "manifest.json"))["files"])
match_ids = sorted(manifest["file"].str.extract(r"(J03W\w+)\.xml")[0].dropna().unique())

def load_match(mid):
    path = OUT / f"{mid}.parquet"
    if path.exists():
        return pd.read_parquet(path)

    ds = sportec.load_tracking(
        raw_data=next(RAW.glob(f"DFL_*positions*{mid}.xml")),
        meta_data=next(RAW.glob(f"DFL_*matchinformation*{mid}.xml")),
        coordinates="sportec", only_alive=False,
    )
    df = ds.to_df()
    home, away = ds.metadata.teams
    side = {p.player_id: g for t, g in [(home, "H"), (away, "A")] for p in t.players}

    label = np.where(df["ball_state"].eq("dead"), "D", np.where(df["ball_owning_team_id"].eq(home.team_id), "H", "A"))
    players = {f"{side[c[:-2]]}_{c}": df[c] for c in df.columns if c[:-2] in side and c[-2:] in ("_x", "_y")}

    out = pd.concat([
        pd.DataFrame({"frame": df["frame_id"], "period": df["period_id"], "label": label,
                      "owner": np.where(df["ball_owning_team_id"].eq(home.team_id), "H", "A")}),
        df[["ball_x", "ball_y", "ball_z"]],
        pd.DataFrame(players),
    ], axis=1)
    out = out.astype({c: "float32" for c in out.columns[4:]}).astype({"label": "category", "owner": "category"})
    out.to_parquet(path)
    return out

matches = {}
for mid in match_ids:
    t0 = time.time()
    matches[mid] = load_match(mid)
    print(f"{mid}: {matches[mid].shape} in {time.time() - t0:.0f} s")

J03WMX: (145967, 71) in 47 s


J03WN1: (141561, 65) in 44 s


J03WOH: (137214, 71) in 42 s


J03WOY: (142536, 71) in 43 s


J03WPY: (146211, 69) in 44 s


J03WQQ: (142345, 69) in 44 s


J03WR9: (146810, 69) in 45 s


## 2. Kontrolle

Je Spiel: Dauer, Anteil der Labels und Anteil der Frames ohne Ball. Im DFL-Tracking ist der Ball immer vorhanden, deshalb wird er in Notebook 02 künstlich verschlechtert.

In [3]:
pd.DataFrame({mid: {"Minuten": len(df) / FPS / 60, **df["label"].value_counts(normalize=True).to_dict(),
                    "Ball fehlt": df["ball_x"].isna().mean()} for mid, df in matches.items()}).T.round(3)

,Minuten,D,A,H,Ball fehlt
J03WMX,97.311,0.415,0.317,0.268,0.0
J03WN1,94.374,0.517,0.226,0.257,0.0
J03WOH,91.476,0.476,0.239,0.285,0.0
J03WOY,95.024,0.465,0.270,0.265,0.0
J03WPY,97.474,0.443,0.207,0.350,0.0
J03WQQ,94.897,0.407,0.332,0.260,0.0
J03WR9,97.873,0.419,0.218,0.363,0.0


**Ergebnis:** Sieben vollständige Spiele mit dichten Frame-Labels. Sie sind die Grundlage für die Degradation in Notebook 02.